In [1]:
import sys
sys.path.append("../..")

import os
os.environ['TF_GPU_ALLOCATOR'] = "cuda_malloc_async"

import pandas as pd
import numpy as np
import requests
import tensorflow as tf

from PIL import Image

from pu.feature_extractors.extractors import ViTExtractor, AutoencoderExtractor
from pu.data.loaders import CSVLoader, SingleCSVLoader, SingleCSVWithTestLoader, FullCSVLoader
from pu.data.pu_builder import build_pu_data, pn_test_split

from datasets import Dataset
from transformers import CLIPProcessor, TFCLIPModel, TFCLIPVisionModel, AutoTokenizer

from sklearn.model_selection import train_test_split

2024-07-18 10:26:38.919592: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-07-18 10:26:38.941331: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-07-18 10:26:38.941349: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-07-18 10:26:38.941366: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-07-18 10:26:38.945929: I tensorflow/core/platform/cpu_feature_g

In [2]:
# Load datasets
def get_datasets():

    dataset_params = {
        'ava': ['/srv/PU-dataset/unlabeled.csv', 'id', '/srv/PU-dataset/dataset_unlabeled'],
        'aadb_train': ['/srv/aadb/train.csv', 'path', '/srv/aadb'],
        'aadb_val': ['/srv/aadb/validation.csv', 'path', '/srv/aadb'],
        'aadb_test': ['/srv/aadb/testnew.csv', 'path', '/srv/aadb'],
        'laion_aes': ['/srv/PU-dataset/positive.csv', 'path', '/srv/PU-dataset/dataset_positive']
    }

    score_columns = ['VotesMean', 'label', 'label', 'label', 'AESTHETIC_SCORE']

    all_datasets = {}

    for dataset, score_col in zip(dataset_params, score_columns):
        loader = FullCSVLoader(*dataset_params[dataset])

        path_col = dataset_params[dataset][1]
        data = loader.load_data()
        data = data.rename(columns={path_col: 'path'})

        all_datasets[dataset] = data

    return all_datasets

datasets = get_datasets()

In [3]:
# Setup train-test splits
def ava_splits(datasets_dict, quantile):
    dataset = datasets_dict[f"ava"]
    
    X_train, X_val, X_test, y_train, y_val, y_test, y_test_pu = build_pu_data(
        dataset,
        frac=1.0,
        move_to_unlabeled_frac=0.0,
        val_split=0.2,
        val_split_positive='same',
        reliable_positive_fn=lambda row, df: row['VotesMean'] > quantile,
        positive_fn=lambda row, df: row['VotesMean'] >= 5.0,
        test_frac=0.2,
        input_mode='images',
        random_state=1234
    )

    return X_train, X_val, X_test, y_train, y_val, y_test, y_test_pu

def aadb_splits(datasets_dict, quantile):
    dataset_train = datasets_dict[f"aadb_train"]
    dataset_val = datasets_dict[f"aadb_val"]
    dataset_test = datasets_dict[f"aadb_test"]
    
    X_train, _, _, y_train, _, _, _ = build_pu_data(
        dataset_train,
        frac=1.0,
        move_to_unlabeled_frac=0.0,
        val_split=0,
        val_split_positive='same',
        reliable_positive_fn=lambda row, df: row['label'] > quantile,
        positive_fn=lambda row, df: row['label'] >= 0.5,
        test_frac=0,
        input_mode='images',
        random_state=1234
    )

    _, X_val, _, _, y_val, _, _ = build_pu_data(
        dataset_val,
        frac=1.0,
        move_to_unlabeled_frac=0.0,
        val_split=1.0,
        val_split_positive='same',
        reliable_positive_fn=lambda row, df: row['label'] > quantile,
        positive_fn=lambda row, df: row['label'] >= 0.5,
        test_frac=0,
        input_mode='images',
        random_state=1234
    )

    _, _, X_test, y_test, y_test_pu = pn_test_split(
        dataset_test, 
        lambda row, df: row['label'] > quantile, 
        lambda row, df: row['label'] >= 0.5, 
        1.0, 
        input_mode='images',
        random_state=1234
    )

    return X_train, X_val, X_test, y_train, y_val, y_test, y_test_pu

# LAION-AES 6.5 is considered to only contain highly-aesthetic images. No need for unlabeled examples.
def laion_splits(datasets_dict, quantile):
    dataset = datasets_dict[f"laion_aes"]

    X_train, X_val, X_test, y_train, y_val, y_test, y_test_pu = build_pu_data(
        dataset,
        frac=1.0,
        move_to_unlabeled_frac=0,
        val_split=0.2,
        val_split_positive='same',
        reliable_positive_fn=lambda row, df: row['AESTHETIC_SCORE'] > quantile,
        positive_fn=lambda row, df: row['AESTHETIC_SCORE'] >= quantile,
        test_frac=0.2,
        input_mode='images',
        random_state=1234
    )

    # A quantile can be used to remove images below a certain threshold. This can
    # help limiting the size of LAION for its experiments
    X_train = X_train[y_train == 1]
    y_train = y_train[y_train == 1]

    X_val = X_val[y_val == 1]
    y_val = y_val[y_val == 1]

    X_test = X_test[y_test == 1]
    y_test = y_test[y_test == 1]

    return X_train, X_val, X_test, y_train, y_val, y_test, y_test_pu

def get_laion_train_func(train_ds_func):
    def train_func(datasets_dict, quantile):
        laion_train, laion_val, laion_test, _, _, _, _ = laion_splits(datasets_dict, quantile)
        laion_full = np.concatenate([laion_train, laion_val, laion_test], axis=0)
        
        other_train, other_val, _, _, _, _, _ = train_ds_func(datasets_dict, quantile)
        other_full = np.concatenate([other_train, other_val], axis=0)
        
        ds_full = np.concatenate([laion_full, other_full], axis=0)
        labels = np.concatenate([np.ones(len(laion_full)), np.zeros(len(other_full))])
        X_train, X_val, y_train, y_val = train_test_split(ds_full, labels, test_size=0.2, random_state=1234, shuffle=True, stratify=labels)
	
        return X_train, X_val, None, y_train, y_val, None, None
	
    return train_func

X_train, X_val, _, y_train, y_val, _, _ = get_laion_train_func(ava_splits)(datasets, 5.0)
_, _, X_test, _, _, y_test, y_test_pu = ava_splits(datasets, 10.0)

In [4]:
# Load model and setup pipeline
vit_model = TFCLIPModel.from_pretrained('openai/clip-vit-large-patch14')

def clip_preprocessor_tf(image):
    # Resize image and center crop
    shape = tf.shape(image)
    height, width = shape[0], shape[1]        
    target_shape = ((tf.cast(224 * height / width, tf.int32), 224) if width < height else (224, tf.cast(224 * width / height, tf.int32)))
    image = tf.image.resize(image, target_shape, method=tf.image.ResizeMethod.BICUBIC)
    image = tf.keras.layers.CenterCrop(224,224)(image)

    # Rescale and normalize image
    image /= 255.0
    image = (image - tf.constant([0.48145466, 0.4578275, 0.40821073])) / tf.constant([0.26862954, 0.26130258, 0.27577711])

    return tf.transpose(image, [2, 0, 1])

#Setup pipelines
def parse_image(filename, label):
    image = tf.io.read_file(filename)
    image = tf.io.decode_jpeg(image, channels=3)
    image = clip_preprocessor_tf(image)    
    
    return image, label

train_dataset = tf.data.Dataset.from_tensor_slices((X_train[:8], y_train[:8])) \
    .shuffle(10000, seed=1234) \
    .map(parse_image) \
    .batch(8) \
    .prefetch(-1)

2024-07-18 10:26:55.032219: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-07-18 10:26:55.035021: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-07-18 10:26:55.035093: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

In [5]:
# Create model and loss function
def create_nn_pu_loss(positive_prior, loss_fn):
    def nn_pu_loss(y_true, y_pred):
        positive_examples = tf.squeeze(tf.gather(y_pred, tf.where(y_true == 1)))
        unlabeled_examples = tf.squeeze(tf.gather(y_pred, tf.where(y_true == 0)))

        positive_positive_risk = positive_prior * tf.reduce_mean(loss_fn(tf.ones_like(positive_examples), positive_examples))
        unlabeled_negative_risk = tf.reduce_mean(loss_fn(tf.zeros_like(unlabeled_examples), unlabeled_examples))
        positive_negative_risk = positive_prior * tf.reduce_mean(loss_fn(tf.zeros_like(positive_examples), positive_examples))

        loss = positive_positive_risk + tf.math.maximum(0.0, unlabeled_negative_risk - positive_negative_risk)

        return loss
    
    return nn_pu_loss

inputs = tf.keras.Input(shape=(3,224,224))
vit_features = vit_model.get_image_features(inputs, training=True)
output = tf.keras.layers.Dense(1, activation="sigmoid")(vit_features)

model = tf.keras.Model(inputs=inputs, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6),
    loss=create_nn_pu_loss(0.7, tf.keras.losses.BinaryCrossentropy())
)

In [7]:
# Setup LoRA for finetuning
import math

class LoraLayer(tf.keras.layers.Layer):
    def __init__(
        self,
        original_layer,
        rank=8,
        alpha=32,
        trainable=False,
        **kwargs,
    ):
        # We want to keep the name of this layer the same as the original
        # dense layer.
        original_layer_config = original_layer.get_config()
        name = original_layer_config["name"]

        kwargs.pop("name", None)

        super().__init__(name=name, trainable=trainable, **kwargs)

        self.rank = rank
        self.alpha = alpha

        self._scale = alpha / rank

        # Layers.

        # Original dense layer.
        self.original_layer = original_layer
        # No matter whether we are training the model or are in inference mode,
        # this layer should be frozen.
        self.original_layer.trainable = False

        # LoRA dense layers.
        self.A = tf.keras.layers.Dense(
            units=rank,
            use_bias=False,
            # Note: the original paper mentions that normal distribution was
            # used for initialization. However, the official LoRA implementation
            # uses "Kaiming/He Initialization".
            kernel_initializer=tf.keras.initializers.VarianceScaling(
                scale=math.sqrt(5), mode="fan_in", distribution="uniform"
            ),
            trainable=trainable,
            name=f"lora_A",
        )

        # Hack: change EinsumDense by a regular dense. Hopefully this works...
        self.B = tf.keras.layers.Dense(
            units=original_layer.units,
            kernel_initializer="zeros",
            trainable=trainable,
            name=f"lora_B",
        )

    def call(self, inputs):
        original_output = self.original_layer(inputs)
        if self.trainable:
            # If we are fine-tuning the model, we will add LoRA layers' output
            # to the original layer's output.
            lora_output = self.B(self.A(inputs)) * self._scale
            return original_output + lora_output

        # If we are in inference mode, we "merge" the LoRA layers' weights into
        # the original layer's weights - more on this in the text generation
        # section!
        return original_output


In [8]:
# Replace original layers by LoRAs
for encoder_layer in model.layers[1].encoder.layers:
    attention = encoder_layer.self_attn

    attention.q_proj = LoraLayer(
        attention.q_proj,
        rank=4,
        alpha=32.0,
        trainable=True,
    )

    attention.v_proj = LoraLayer(
        attention.v_proj,
        rank=4,
        alpha=32.0,
        trainable=True,
    )

In [10]:
model.fit(train_dataset, epochs=200)

Epoch 1/200


2024-07-18 10:28:53.163533: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:442] Loaded cuDNN version 8600
2024-07-18 10:28:54.488647: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x773980144c60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-07-18 10:28:54.488670: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 3080 Ti, Compute Capability 8.6
2024-07-18 10:28:54.494667: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-07-18 10:28:54.558671: I ./tensorflow/compiler/jit/device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1/1 [==============================] - 57s 57s/step - loss: 0.9167
Epoch 2/200
1/1 [==============================] - 0s 370ms/step - loss: 0.6428
Epoch 3/200
1/1 [==============================] - 0s 351ms/step - loss: 0.7397
Epoch 4/200
1/1 [==============================] - 0s 350ms/step - loss: 0.5079
Epoch 5/200
1/1 [==============================] - 0s 348ms/step - loss: 0.2108
Epoch 6/200
1/1 [==============================] - 0s 347ms/step - loss: 0.1530
Epoch 7/200
1/1 [==============================] - 0s 346ms/step - loss: 0.1916
Epoch 8/200
1/1 [==============================] - 0s 350ms/step - loss: 0.1237
Epoch 9/200
1/1 [==============================] - 0s 351ms/step - loss: 0.3509
Epoch 10/200
1/1 [==============================] - 0s 347ms/step - loss: 0.3086
Epoch 11/200
1/1 [==============================] - 0s 347ms/step - loss: 0.1403
Epoch 12/200
1/1 [==============================] - 0s 350ms/step - loss: 0.1153
Epoch 13/200
1/1 [==============================] 


KeyboardInterrupt

